In [15]:
"""
GLOBAL SENSITIVITY ANALYSIS (SOBOL) - THERMAL / HYGROSCOPIC MISMATCH INDEX
CASE A - EXPOSURE: Thickness, Time and dT are VARIABLE; all other inputs are held fixed.
==========================================================================

MODEL CHAIN
-----------
  0. Arrhenius        D = D0 * exp(-Ea / (R*T)),  T = T_ref + dT
  1. Fickian uptake   Mt/Minf = 1 - (8/pi^2) * SUM 1/(2n+1)^2
                                 * exp[-(2n+1)^2 pi^2 D t / h^2]
  2. Moisture conc.   C_moist [kg/m3] = (Mt/Minf) * C_sat [mol/m3] * M_water [kg/mol]
  3. Hygroscopic strain  eps_h = beta [m3/kg] * C_moist [kg/m3]
  4. Thermal strain      eps_T = (CTE_laminate - CTE_copper) * dT
  5. Mismatch index      sigma = |E_eff / (1 - nu_eff) * (eps_T + eps_h)|   [MPa]

METHOD
------
Saltelli cross-sampling on a scrambled Sobol' sequence with the
Jansen / Saltelli-2010 estimators (as in SALib). Needs numpy + scipy
(matplotlib optional for plots, openpyxl for the Excel workbooks).
Model runs per case = N * (k + 2), k = number of variable inputs.

Run:  python3 sobol_A_exposure.py [N]
"""

import math
import sys
import warnings

import numpy as np
from scipy.stats import qmc


# ======================================================================
# 1. INPUT DEFINITIONS
# ======================================================================
# VARIABLE inputs (sampled):  name: (unit, [lower, upper], distribution)
INPUTS = {
    "Time":      ("s",  [8.64e4, 3.1536e7], "logunif"),   # 1 day - 1 year
    "Thickness": ("m",  [0.0002, 0.0032],  "unif"),      # 0.2 - 3.2 mm
    "dT":        ("K",  [10.0, 100.0],     "unif"),
}

# CONSTANT inputs (fixed values - *** edit to your material ***)
CONSTANTS = {
    "C_sat":   222.04,     # mol/m^3   saturation concentration
    "beta_h":  1.4e-4,   # m^3/kg    coefficient of hygroscopic strain
    "CTE_lam": 13.5e-6,    # 1/K       laminate CTE
    "CTE_cu":  17.0e-6,    # 1/K       copper CTE
    "E_eff":   24.056e9,    # Pa        effective modulus
    "nu_eff":  0.182,     # -         effective Poisson's ratio
    "M_water": 0.018015,   # kg/mol    molar mass of water
    # Arrhenius diffusivity  D = D0 * exp(-Ea / (R * T)),  T = T_ref + dT
    "D0":      6.31e-7,    # m^2/s     pre-exponential factor  
    "Ea":      37560.0,    # J/mol     activation energy       
    "T_ref":   298.15,     # K         reference temperature (25 C) that dT is added to
    "R":       8.314,      # J/(mol K) gas constant
}
CONST_UNITS = {
    "C_sat": "mol/m^3", "beta_h": "m^3/kg", "CTE_lam": "1/K", "CTE_cu": "1/K",
    "E_eff": "Pa", "nu_eff": "-", "M_water": "kg/mol", "D0": "m^2/s",
    "Ea": "J/mol", "T_ref": "K", "R": "J/(mol K)",
}

CASES = {
    "A_exposure": list(INPUTS),
}


def make_problem(vary):
    problem = {
        "names": list(vary),
        "units": [INPUTS[n][0] for n in vary],
        "bounds": [INPUTS[n][1] for n in vary],
        "dists": [INPUTS[n][2] for n in vary],
    }
    return problem, dict(CONSTANTS)


# ======================================================================
# 2. MODEL
# ======================================================================
def moisture_ratio(D, h, t, n_terms=60):
    """Mt/Minf, vectorised. D, h, t are arrays of equal length."""
    m = 2 * np.arange(n_terms) + 1
    arg = -(m[None, :] ** 2) * math.pi ** 2 * (D * t / h ** 2)[:, None]
    series = np.sum(np.exp(arg) / m[None, :] ** 2, axis=1)
    return np.clip(1.0 - (8.0 / math.pi ** 2) * series, 0.0, 1.0)


def model(X, names, fixed):
    """X columns follow `names`; every other quantity is a constant."""
    n = X.shape[0]

    def g(name):
        if name in names:
            return X[:, names.index(name)]
        return np.full(n, fixed[name])

    t, h, dT = g("Time"), g("Thickness"), g("dT")
    C_sat, beta = g("C_sat"), g("beta_h")
    CTE_lam = g("CTE_lam")
    E_eff, nu_eff = g("E_eff"), g("nu_eff")

    # Arrhenius temperature-dependent diffusivity
    T = CONSTANTS["T_ref"] + dT
    D = CONSTANTS["D0"] * np.exp(-CONSTANTS["Ea"] / (CONSTANTS["R"] * T))

    Mt_Minf = moisture_ratio(D, h, t)
    Fo = D * t / h ** 2
    C_moist = Mt_Minf * C_sat * CONSTANTS["M_water"]
    eps_h = beta * C_moist
    eps_T = (CTE_lam - CONSTANTS["CTE_cu"]) * dT
    sigma = np.abs(E_eff / (1.0 - nu_eff) * (eps_T + eps_h)) / 1e6

    return {"D": D, "Fo": Fo, "Mt_Minf": Mt_Minf, "C_moist": C_moist,
            "eps_h": eps_h, "eps_T": eps_T, "sigma": sigma}


Y_KEY = "sigma"
Y_LABEL = "Mismatch stress index (MPa)"

OUTPUT_TABLE = [("D", "D (m^2/s)"),
                ("Fo", "Fourier number (-)"),
                ("Mt_Minf", "Mt/Minf (-)"),
                ("C_moist", "C_moist (kg/m^3)"),
                ("eps_h", "Hygroscopic strain (-)"),
                ("eps_T", "Thermal strain (-)"),
                ("sigma", "Mismatch index (MPa)")]


# ======================================================================
# 3. SALTELLI SAMPLING
# ======================================================================
def saltelli_sample(bounds, N, dists=None, seed=12345):
    """Rows ordered [A | B | AB_0..AB_{k-1}]. Returns (X_all, k)."""
    k = len(bounds)
    if dists is None:
        dists = ["unif"] * k

    sampler = qmc.Sobol(d=2 * k, scramble=True, seed=seed)
    base = sampler.random(N)

    lo = np.array([b[0] for b in bounds], dtype=float)
    hi = np.array([b[1] for b in bounds], dtype=float)
    log_mask = np.array([d == "logunif" for d in dists])

    def scale(U):
        Xs = lo + U * (hi - lo)
        if log_mask.any():
            llo, lhi = np.log(lo[log_mask]), np.log(hi[log_mask])
            Xs[:, log_mask] = np.exp(llo + U[:, log_mask] * (lhi - llo))
        return Xs

    A = scale(base[:, :k])
    B = scale(base[:, k:])

    blocks = [A, B]
    for i in range(k):
        M = A.copy()
        M[:, i] = B[:, i]
        blocks.append(M)
    return np.vstack(blocks), k


def block_labels(N, k, names):
    labs = ["A"] * N + ["B"] * N
    for i in range(k):
        labs += [f"AB_{names[i]}"] * N
    return np.array(labs, dtype=object)


# ======================================================================
# 4. SOBOL ESTIMATORS
# ======================================================================
def split_outputs(Y, N, k):
    yA = Y[0:N]
    yB = Y[N:2 * N]
    yAB = [Y[(2 + i) * N:(3 + i) * N] for i in range(k)]
    return yA, yB, yAB


def sobol_indices(Y, N, k, n_boot=500, seed=7):
    """First- and total-order indices with bootstrap 95% CIs."""
    yA, yB, yAB = split_outputs(Y, N, k)
    rng = np.random.default_rng(seed)

    def _s1(a, ab, b, idx):
        V = np.var(np.r_[a[idx], b[idx]], ddof=0)
        return np.mean(b[idx] * (ab[idx] - a[idx])) / V

    def _st(a, ab, b, idx):
        V = np.var(np.r_[a[idx], b[idx]], ddof=0)
        return 0.5 * np.mean((a[idx] - ab[idx]) ** 2) / V

    full = np.arange(N)
    S1 = np.array([_s1(yA, yAB[i], yB, full) for i in range(k)])
    ST = np.array([_st(yA, yAB[i], yB, full) for i in range(k)])

    b1 = np.empty((n_boot, k))
    bT = np.empty((n_boot, k))
    for r in range(n_boot):
        idx = rng.integers(0, N, N)
        for i in range(k):
            b1[r, i] = _s1(yA, yAB[i], yB, idx)
            bT[r, i] = _st(yA, yAB[i], yB, idx)

    def ci(arr):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            return (np.nanpercentile(arr, 2.5, axis=0),
                    np.nanpercentile(arr, 97.5, axis=0))

    return {"S1": S1, "S1_ci": ci(b1), "ST": ST, "ST_ci": ci(bT)}


# ======================================================================
# 5. REPORTING
# ======================================================================
def report(res, names, label):
    lo1, hi1 = res["S1_ci"]
    loT, hiT = res["ST_ci"]

    print("\n" + "=" * 72)
    print(f" SOBOL INDICES  -  {label}")
    print("=" * 72)
    print(f" {'Variable':<14} {'S1':>9} {'95% CI':>20} {'ST':>9} {'95% CI':>20}")
    print(" " + "-" * 70)
    for i, n in enumerate(names):
        print(f" {n:<14} {res['S1'][i]:>9.4f} "
              f"[{lo1[i]:>7.4f},{hi1[i]:>8.4f}] "
              f"{res['ST'][i]:>9.4f} [{loT[i]:>7.4f},{hiT[i]:>8.4f}]")
    print(" " + "-" * 70)
    print(f" {'Sum S1':<14} {res['S1'].sum():>9.4f}"
          f"{'':>20} {'Sum ST':<9} {res['ST'].sum():>9.4f}")
    print(f" Interaction content (sum ST - sum S1) : "
          f"{res['ST'].sum() - res['S1'].sum():.4f}")
    order = np.argsort(-res["ST"])
    print("\n Ranking by total-order influence: "
          + " > ".join(names[i] for i in order))


def summarise_outputs(out):
    print("\n" + "=" * 72)
    print(" MONTE CARLO OUTPUT SUMMARY (all sampled combinations)")
    print("=" * 72)
    print(f" {'Quantity':<24} {'min':>12} {'mean':>12} {'max':>12} {'std':>12}")
    print(" " + "-" * 72)
    for key, lab in OUTPUT_TABLE:
        v = out[key]
        print(f" {lab:<24} {v.min():>12.4g} {v.mean():>12.4g} "
              f"{v.max():>12.4g} {v.std():>12.4g}")


# ======================================================================
# 6. OPTIONAL PLOTS
# ======================================================================
def make_plots(res, names, title, fname):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("\n matplotlib not available - skipping plots.")
        return None

    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.bar(x - 0.2, res["S1"], 0.4, label="S1 (first-order)")
    ax.bar(x + 0.2, res["ST"], 0.4, label="ST (total-order)")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha="right")
    ax.set_ylabel("Sobol index")
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)
    print(f"\n Plot saved to {fname}")
    return fname


# ======================================================================
# 7. EXCEL EXPORT
# ======================================================================
def save_excel(X, out, res, problem, fixed, N, case_name, png, fname):
    try:
        import openpyxl
        from openpyxl.styles import Font, Alignment, PatternFill
        from openpyxl.utils import get_column_letter
    except ImportError:
        print("\n openpyxl not available - skipping Excel export.")
        return None

    names = problem["names"]
    k = len(names)

    ARIAL = "Arial"
    head_font = Font(name=ARIAL, bold=True, color="FFFFFF")
    head_fill = PatternFill("solid", fgColor="44546A")
    title_font = Font(name=ARIAL, bold=True, size=12)
    body = Font(name=ARIAL)
    bold = Font(name=ARIAL, bold=True)

    wb = openpyxl.Workbook()

    def write_header(ws, row, headers, widths=None):
        for c, h in enumerate(headers, start=1):
            cell = ws.cell(row=row, column=c, value=h)
            cell.font = head_font
            cell.fill = head_fill
            cell.alignment = Alignment(horizontal="center", wrap_text=True)
        if widths:
            for c, w in enumerate(widths, start=1):
                ws.column_dimensions[get_column_letter(c)].width = w

    # ---------------- Setup ----------------------------------------
    ws = wb.active
    ws.title = "Setup"
    ws["A1"] = f"Global sensitivity analysis - mismatch index - case {case_name}"
    ws["A1"].font = title_font
    ws["A2"] = "sigma = abs( E_eff/(1-nu_eff) * (eps_T + eps_h) )"
    ws["A2"].font = Font(name=ARIAL, italic=True)
    ws["A3"] = "D = D0 * exp(-Ea / (R * (T_ref + dT)))"
    ws["A3"].font = Font(name=ARIAL, italic=True)

    ws["A4"] = "Variable inputs (varied simultaneously)"
    ws["A4"].font = bold
    write_header(ws, 5, ["Variable", "Units", "Lower bound", "Upper bound",
                         "Distribution"], [24, 12, 16, 16, 14])
    r = 6
    for n, u, b, d in zip(names, problem["units"], problem["bounds"],
                          problem["dists"]):
        for c, v in enumerate([n, u, float(b[0]), float(b[1]), d], start=1):
            ws.cell(row=r, column=c, value=v).font = body
        r += 1

    r += 1
    ws.cell(row=r, column=1, value="Fixed inputs (held constant)").font = bold
    r += 1
    write_header(ws, r, ["Parameter", "Value", "Units"])
    r += 1
    for key, val in fixed.items():
        ws.cell(row=r, column=1, value=key).font = body
        ws.cell(row=r, column=2, value=float(val)).font = body
        ws.cell(row=r, column=3, value=CONST_UNITS[key]).font = body
        r += 1

    r += 1
    write_header(ws, r, ["Sampling", "Value", "Note"])
    ws.column_dimensions["C"].width = 44
    r += 1
    for lab, val, note in [
        ("Base sample size N", N, "power of two, scrambled Sobol' sequence"),
        ("Number of variables k", k, ", ".join(names)),
        ("Model evaluations", N * (k + 2), "N*(k+2), Saltelli cross-sampling"),
        ("Bootstrap replicates", 500, "for the 95% confidence intervals"),
        ("Series terms", 60, "terms kept in the Fickian series solution"),
    ]:
        ws.cell(row=r, column=1, value=lab).font = body
        ws.cell(row=r, column=2, value=val).font = body
        ws.cell(row=r, column=3, value=note).font = body
        r += 1

    # ---------------- Samples --------------------------------------
    ws = wb.create_sheet("Samples")
    out_cols = [("D", "Diffusivity D (m^2/s)"),
                ("Fo", "Fourier number D*t/h^2 (-)"),
                ("Mt_Minf", "Mt/Minf (-)"),
                ("C_moist", "Moisture conc. (kg/m^3)"),
                ("eps_h", "Hygroscopic strain (-)"),
                ("eps_T", "Thermal strain (-)"),
                ("sigma", "Mismatch index (MPa)")]
    headers = (["Run"] + [f"{n} ({u})" for n, u in zip(names, problem["units"])]
               + [lab for _, lab in out_cols] + ["Saltelli block"])
    write_header(ws, 1, headers, [8] + [16] * (len(headers) - 2) + [16])

    labels = block_labels(N, k, names)
    data = np.column_stack([X] + [out[key] for key, _ in out_cols])
    for i in range(data.shape[0]):
        row = i + 2
        ws.cell(row=row, column=1, value=i + 1)
        for c in range(data.shape[1]):
            ws.cell(row=row, column=c + 2, value=float(data[i, c]))
        ws.cell(row=row, column=data.shape[1] + 2, value=labels[i])

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{data.shape[0] + 1}"
    n_rows = data.shape[0]

    # ---------------- Output summary -------------------------------
    ws = wb.create_sheet("Output summary")
    ws["A1"] = "Statistics over every sampled combination (live formulas)"
    ws["A1"].font = title_font
    write_header(ws, 3, ["Quantity", "Min", "Mean", "Max", "Std dev"],
                 [32, 16, 16, 16, 16])
    quantities = [(f"{n} ({u})", get_column_letter(2 + i))
                  for i, (n, u) in enumerate(zip(names, problem["units"]))]
    quantities += [(lab, get_column_letter(k + 2 + j))
                   for j, (_, lab) in enumerate(out_cols)]
    r = 4
    for lab, col in quantities:
        rng_ = f"Samples!${col}$2:${col}${n_rows + 1}"
        ws.cell(row=r, column=1, value=lab).font = body
        ws.cell(row=r, column=2, value=f"=MIN({rng_})").font = body
        ws.cell(row=r, column=3, value=f"=AVERAGE({rng_})").font = body
        ws.cell(row=r, column=4, value=f"=MAX({rng_})").font = body
        ws.cell(row=r, column=5, value=f"=STDEV({rng_})").font = body
        r += 1

    # ---------------- Sobol S1 / ST --------------------------------
    lo1, hi1 = res["S1_ci"]
    loT, hiT = res["ST_ci"]
    ws = wb.create_sheet("Sobol S1 ST")
    ws["A1"] = f"First- and total-order Sobol indices for {Y_LABEL}"
    ws["A1"].font = title_font
    write_header(ws, 3, ["Variable", "S1", "S1 95% low", "S1 95% high",
                         "ST", "ST 95% low", "ST 95% high",
                         "ST - S1 (interaction share)"],
                 [16, 12, 14, 14, 12, 14, 14, 24])
    for i, n in enumerate(names):
        r = 4 + i
        vals = [n, res["S1"][i], lo1[i], hi1[i], res["ST"][i], loT[i], hiT[i]]
        for c, v in enumerate(vals, start=1):
            ws.cell(row=r, column=c,
                    value=v if c == 1 else float(v)).font = body
        ws.cell(row=r, column=8, value=f"=E{r}-B{r}").font = body

    tot = 4 + k
    ws.cell(row=tot, column=1, value="Sum").font = bold
    ws.cell(row=tot, column=2, value=f"=SUM(B4:B{tot - 1})").font = bold
    ws.cell(row=tot, column=5, value=f"=SUM(E4:E{tot - 1})").font = bold
    ws.cell(row=tot, column=8, value=f"=E{tot}-B{tot}").font = bold
    ws.cell(row=tot + 2, column=1,
            value="Sum ST - Sum S1 is the total interaction content; "
                  "Sum S1 well below 1 means a strongly non-additive model."
            ).font = Font(name=ARIAL, italic=True)

    # ---------------- Charts ---------------------------------------
    if png:
        try:
            from openpyxl.drawing.image import Image as XLImage
            wsc = wb.create_sheet("Charts")
            wsc["A1"] = "S1 / ST bar chart"
            wsc["A1"].font = title_font
            wsc.add_image(XLImage(png), "A3")
        except Exception as exc:
            print(f" (chart image not embedded: {exc})")

    # ---------------- How to read ----------------------------------
    ws = wb.create_sheet("How to read")
    ws.column_dimensions["A"].width = 12
    ws.column_dimensions["B"].width = 100
    ws["A1"] = "How to read this workbook"
    ws["A1"].font = title_font
    notes = [
        ("S1", "Fraction of output variance removed by fixing that variable "
               "alone. High S1 = strong direct driver."),
        ("ST", "S1 plus every interaction the variable takes part in. "
               "ST ~ 0 means the variable can be fixed anywhere in range."),
        ("ST - S1", "Large => the variable acts mainly through interactions."),
        ("Sum S1", "Well below 1 => the model is strongly non-additive."),
        ("Fixed", "Inputs not varied in this case are held at the values on "
                  "the Setup sheet; indices are conditional on those values."),
        ("Samples", "Every model evaluation: sampled inputs, intermediate "
                    "quantities and the mismatch index, plus the Saltelli "
                    "block (A, B, AB_i) each row came from."),
    ]
    r = 3
    for a, b in notes:
        ws.cell(row=r, column=1, value=a).font = bold
        cell = ws.cell(row=r, column=2, value=b)
        cell.font = body
        cell.alignment = Alignment(wrap_text=True, vertical="top")
        r += 1

    wb.save(fname)
    print(f" Excel workbook saved to {fname}  "
          f"({n_rows:,} sampled runs on the Samples sheet)")
    return fname


# ======================================================================
# 8. RUN ONE CASE
# ======================================================================
def run_case(case_name, vary, N):
    problem, fixed = make_problem(vary)
    names, k = problem["names"], len(problem["names"])

    print("\n" + "#" * 72)
    print(f" CASE {case_name}   (k = {k} variable inputs)")
    print("#" * 72)
    print("\n Variable inputs:")
    for n, u, b, d in zip(names, problem["units"], problem["bounds"],
                          problem["dists"]):
        print(f"   {n:<12} {b[0]:>11.4g}  to {b[1]:>11.4g}   [{u}]  ({d})")
    print("\n Constants (held fixed):")
    for key, val in fixed.items():
        print(f"   {key:<12} {val:>11.4g}   [{CONST_UNITS[key]}]")
    print(f"\n Model evaluations: N*(k+2) = {N}*({k + 2}) = {N * (k + 2):,}")

    X, _ = saltelli_sample(problem["bounds"], N, problem["dists"])
    out = model(X, names, fixed)
    summarise_outputs(out)

    res = sobol_indices(out[Y_KEY], N, k)
    report(res, names, f"case {case_name}: {Y_LABEL}")

    png = make_plots(res, names, f"Sensitivity - mismatch index ({case_name})",
                     f"sobol_indices_{case_name}.png")
    save_excel(X, out, res, problem, fixed, N, case_name, png,
               f"sobol_mismatch_{case_name}.xlsx")
    return res


# ======================================================================
# 9. MAIN
# ======================================================================
def main():
    print("=" * 72)
    print(" SOBOL SENSITIVITY - THERMAL/HYGROSCOPIC MISMATCH INDEX - CASE A")
    print("=" * 72)

    N = None
    if len(sys.argv) > 1:
        try:
            N = int(sys.argv[1])
        except ValueError:
            N = None
    if N is None:
        try:
            raw = input("\n Base sample size N (power of 2, blank = 4096): ").strip()
            N = int(raw) if raw else 4096
        except (ValueError, EOFError):
            N = 4096
    if N & (N - 1):
        N = 1 << (N - 1).bit_length()
        print(f" -> rounded up to the next power of two: N = {N}")

    for case_name, vary in CASES.items():
        run_case(case_name, vary, N)

    print("\n" + "=" * 72)
    print(" HOW TO READ THIS")
    print("=" * 72)
    print(" S1  fraction of output variance removed by fixing that variable")
    print("     alone. High S1 = strong direct driver.")
    print(" ST  S1 plus every interaction the variable takes part in.")
    print("     ST ~ 0 means the variable can be fixed at any value in range.")
    print(" Indices are conditional on the constant values set in CONSTANTS.")


if __name__ == "__main__":
    main()

 SOBOL SENSITIVITY - THERMAL/HYGROSCOPIC MISMATCH INDEX - CASE A



 Base sample size N (power of 2, blank = 4096):  



########################################################################
 CASE A_exposure   (k = 3 variable inputs)
########################################################################

 Variable inputs:
   Time            8.64e+04  to   3.154e+07   [s]  (logunif)
   Thickness         0.0002  to      0.0032   [m]  (unif)
   dT                    10  to         100   [K]  (unif)

 Constants (held fixed):
   C_sat                222   [mol/m^3]
   beta_h           0.00014   [m^3/kg]
   CTE_lam         1.35e-05   [1/K]
   CTE_cu           1.7e-05   [1/K]
   E_eff          2.406e+10   [Pa]
   nu_eff             0.182   [-]
   M_water          0.01801   [kg/mol]
   D0              6.31e-07   [m^2/s]
   Ea             3.756e+04   [J/mol]
   T_ref              298.1   [K]
   R                  8.314   [J/(mol K)]

 Model evaluations: N*(k+2) = 4096*(5) = 20,480

 MONTE CARLO OUTPUT SUMMARY (all sampled combinations)
 Quantity                          min         mean          max        

In [19]:
"""
GLOBAL SENSITIVITY ANALYSIS (SOBOL) - THERMAL / HYGROSCOPIC MISMATCH INDEX
CASE B - MATERIAL: D, C_sat, beta_h, CTE_lam, E_eff, nu_eff are VARIABLE; Thickness, Time and dT are held fixed.
==========================================================================

MODEL CHAIN
-----------
  1. Fickian uptake   Mt/Minf = 1 - (8/pi^2) * SUM 1/(2n+1)^2
                                 * exp[-(2n+1)^2 pi^2 D t / h^2]
  2. Moisture conc.   C_moist [kg/m3] = (Mt/Minf) * C_sat [mol/m3] * M_water [kg/mol]
  3. Hygroscopic strain  eps_h = beta [m3/kg] * C_moist [kg/m3]
  4. Thermal strain      eps_T = (CTE_laminate - CTE_copper) * dT
  5. Mismatch index      sigma = |E_eff / (1 - nu_eff) * (eps_T + eps_h)|   [MPa]

METHOD
------
Saltelli cross-sampling on a scrambled Sobol' sequence with the
Jansen / Saltelli-2010 estimators (as in SALib). Needs numpy + scipy
(matplotlib optional for plots, openpyxl for the Excel workbooks).
Model runs per case = N * (k + 2), k = number of variable inputs.

Run:  python3 sobol_B_material.py [N]
"""

import math
import sys
import warnings

import numpy as np
from scipy.stats import qmc


# ======================================================================
# 1. INPUT DEFINITIONS
# ======================================================================
# VARIABLE inputs (sampled):  name: (unit, [lower, upper], distribution)
INPUTS = {
    "D":       ("m^2/s",   [2.15e-14, 2.0e-12], "logunif"),
    "C_sat":   ("mol/m^3", [92.5, 366.0],       "unif"),
    "beta_h":  ("m^3/kg",  [1.15e-4, 3.8e-4],   "unif"),
    "CTE_lam": ("1/K",     [8e-6, 23e-6],       "unif"),
    "E_eff":   ("Pa",      [7.5e9, 28.0e9],     "unif"),
    "nu_eff":  ("-",       [0.175, 0.32],       "unif"),
}

# CONSTANT inputs (fixed values - *** edit to your service condition ***)
CONSTANTS = {
    "Time":      4.765e5,   # s       exposure time (about 5.5 days)
    "Thickness": 1.6e-3,    # m       laminate thickness
    "dT":        50.0,      # K       change in temperature
    "CTE_cu":    16.5e-6,   # 1/K     copper CTE
    "M_water":   0.018015,  # kg/mol  molar mass of water
}
CONST_UNITS = {
    "Time": "s", "Thickness": "m", "dT": "K", "CTE_cu": "1/K",
    "M_water": "kg/mol",
}

CASES = {
    "B_material": list(INPUTS),
}


def make_problem(vary):
    problem = {
        "names": list(vary),
        "units": [INPUTS[n][0] for n in vary],
        "bounds": [INPUTS[n][1] for n in vary],
        "dists": [INPUTS[n][2] for n in vary],
    }
    return problem, dict(CONSTANTS)


# ======================================================================
# 2. MODEL
# ======================================================================
def moisture_ratio(D, h, t, n_terms=60):
    """Mt/Minf, vectorised. D, h, t are arrays of equal length."""
    m = 2 * np.arange(n_terms) + 1
    arg = -(m[None, :] ** 2) * math.pi ** 2 * (D * t / h ** 2)[:, None]
    series = np.sum(np.exp(arg) / m[None, :] ** 2, axis=1)
    return np.clip(1.0 - (8.0 / math.pi ** 2) * series, 0.0, 1.0)


def model(X, names, fixed):
    """X columns follow `names`; every other quantity is a constant."""
    n = X.shape[0]

    def g(name):
        if name in names:
            return X[:, names.index(name)]
        return np.full(n, fixed[name])

    t, D, h = g("Time"), g("D"), g("Thickness")
    C_sat, beta = g("C_sat"), g("beta_h")
    CTE_lam, dT = g("CTE_lam"), g("dT")
    E_eff, nu_eff = g("E_eff"), g("nu_eff")

    Mt_Minf = moisture_ratio(D, h, t)
    Fo = D * t / h ** 2
    C_moist = Mt_Minf * C_sat * CONSTANTS["M_water"]
    eps_h = beta * C_moist
    eps_T = (CTE_lam - CONSTANTS["CTE_cu"]) * dT
    sigma = np.abs(E_eff / (1.0 - nu_eff) * (eps_T + eps_h)) / 1e6

    return {"Fo": Fo, "Mt_Minf": Mt_Minf, "C_moist": C_moist,
            "eps_h": eps_h, "eps_T": eps_T, "sigma": sigma}


Y_KEY = "sigma"
Y_LABEL = "Mismatch stress index (MPa)"

OUTPUT_TABLE = [("Fo", "Fourier number (-)"),
                ("Mt_Minf", "Mt/Minf (-)"),
                ("C_moist", "C_moist (kg/m^3)"),
                ("eps_h", "Hygroscopic strain (-)"),
                ("eps_T", "Thermal strain (-)"),
                ("sigma", "Mismatch index (MPa)")]


# ======================================================================
# 3. SALTELLI SAMPLING
# ======================================================================
def saltelli_sample(bounds, N, dists=None, seed=12345):
    """Rows ordered [A | B | AB_0..AB_{k-1}]. Returns (X_all, k)."""
    k = len(bounds)
    if dists is None:
        dists = ["unif"] * k

    sampler = qmc.Sobol(d=2 * k, scramble=True, seed=seed)
    base = sampler.random(N)

    lo = np.array([b[0] for b in bounds], dtype=float)
    hi = np.array([b[1] for b in bounds], dtype=float)
    log_mask = np.array([d == "logunif" for d in dists])

    def scale(U):
        Xs = lo + U * (hi - lo)
        if log_mask.any():
            llo, lhi = np.log(lo[log_mask]), np.log(hi[log_mask])
            Xs[:, log_mask] = np.exp(llo + U[:, log_mask] * (lhi - llo))
        return Xs

    A = scale(base[:, :k])
    B = scale(base[:, k:])

    blocks = [A, B]
    for i in range(k):
        M = A.copy()
        M[:, i] = B[:, i]
        blocks.append(M)
    return np.vstack(blocks), k


def block_labels(N, k, names):
    labs = ["A"] * N + ["B"] * N
    for i in range(k):
        labs += [f"AB_{names[i]}"] * N
    return np.array(labs, dtype=object)


# ======================================================================
# 4. SOBOL ESTIMATORS
# ======================================================================
def split_outputs(Y, N, k):
    yA = Y[0:N]
    yB = Y[N:2 * N]
    yAB = [Y[(2 + i) * N:(3 + i) * N] for i in range(k)]
    return yA, yB, yAB


def sobol_indices(Y, N, k, n_boot=500, seed=7):
    """First- and total-order indices with bootstrap 95% CIs."""
    yA, yB, yAB = split_outputs(Y, N, k)
    rng = np.random.default_rng(seed)

    def _s1(a, ab, b, idx):
        V = np.var(np.r_[a[idx], b[idx]], ddof=0)
        return np.mean(b[idx] * (ab[idx] - a[idx])) / V

    def _st(a, ab, b, idx):
        V = np.var(np.r_[a[idx], b[idx]], ddof=0)
        return 0.5 * np.mean((a[idx] - ab[idx]) ** 2) / V

    full = np.arange(N)
    S1 = np.array([_s1(yA, yAB[i], yB, full) for i in range(k)])
    ST = np.array([_st(yA, yAB[i], yB, full) for i in range(k)])

    b1 = np.empty((n_boot, k))
    bT = np.empty((n_boot, k))
    for r in range(n_boot):
        idx = rng.integers(0, N, N)
        for i in range(k):
            b1[r, i] = _s1(yA, yAB[i], yB, idx)
            bT[r, i] = _st(yA, yAB[i], yB, idx)

    def ci(arr):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            return (np.nanpercentile(arr, 2.5, axis=0),
                    np.nanpercentile(arr, 97.5, axis=0))

    return {"S1": S1, "S1_ci": ci(b1), "ST": ST, "ST_ci": ci(bT)}


# ======================================================================
# 5. REPORTING
# ======================================================================
def report(res, names, label):
    lo1, hi1 = res["S1_ci"]
    loT, hiT = res["ST_ci"]

    print("\n" + "=" * 72)
    print(f" SOBOL INDICES  -  {label}")
    print("=" * 72)
    print(f" {'Variable':<14} {'S1':>9} {'95% CI':>20} {'ST':>9} {'95% CI':>20}")
    print(" " + "-" * 70)
    for i, n in enumerate(names):
        print(f" {n:<14} {res['S1'][i]:>9.4f} "
              f"[{lo1[i]:>7.4f},{hi1[i]:>8.4f}] "
              f"{res['ST'][i]:>9.4f} [{loT[i]:>7.4f},{hiT[i]:>8.4f}]")
    print(" " + "-" * 70)
    print(f" {'Sum S1':<14} {res['S1'].sum():>9.4f}"
          f"{'':>20} {'Sum ST':<9} {res['ST'].sum():>9.4f}")
    print(f" Interaction content (sum ST - sum S1) : "
          f"{res['ST'].sum() - res['S1'].sum():.4f}")
    order = np.argsort(-res["ST"])
    print("\n Ranking by total-order influence: "
          + " > ".join(names[i] for i in order))


def summarise_outputs(out):
    print("\n" + "=" * 72)
    print(" MONTE CARLO OUTPUT SUMMARY (all sampled combinations)")
    print("=" * 72)
    print(f" {'Quantity':<24} {'min':>12} {'mean':>12} {'max':>12} {'std':>12}")
    print(" " + "-" * 72)
    for key, lab in OUTPUT_TABLE:
        v = out[key]
        print(f" {lab:<24} {v.min():>12.4g} {v.mean():>12.4g} "
              f"{v.max():>12.4g} {v.std():>12.4g}")


# ======================================================================
# 6. OPTIONAL PLOTS
# ======================================================================
def make_plots(res, names, title, fname):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("\n matplotlib not available - skipping plots.")
        return None

    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.bar(x - 0.2, res["S1"], 0.4, label="S1 (first-order)")
    ax.bar(x + 0.2, res["ST"], 0.4, label="ST (total-order)")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha="right")
    ax.set_ylabel("Sobol index")
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(fname, dpi=150)
    plt.close(fig)
    print(f"\n Plot saved to {fname}")
    return fname


# ======================================================================
# 7. EXCEL EXPORT
# ======================================================================
def save_excel(X, out, res, problem, fixed, N, case_name, png, fname):
    try:
        import openpyxl
        from openpyxl.styles import Font, Alignment, PatternFill
        from openpyxl.utils import get_column_letter
    except ImportError:
        print("\n openpyxl not available - skipping Excel export.")
        return None

    names = problem["names"]
    k = len(names)

    ARIAL = "Arial"
    head_font = Font(name=ARIAL, bold=True, color="FFFFFF")
    head_fill = PatternFill("solid", fgColor="44546A")
    title_font = Font(name=ARIAL, bold=True, size=12)
    body = Font(name=ARIAL)
    bold = Font(name=ARIAL, bold=True)

    wb = openpyxl.Workbook()

    def write_header(ws, row, headers, widths=None):
        for c, h in enumerate(headers, start=1):
            cell = ws.cell(row=row, column=c, value=h)
            cell.font = head_font
            cell.fill = head_fill
            cell.alignment = Alignment(horizontal="center", wrap_text=True)
        if widths:
            for c, w in enumerate(widths, start=1):
                ws.column_dimensions[get_column_letter(c)].width = w

    # ---------------- Setup ----------------------------------------
    ws = wb.active
    ws.title = "Setup"
    ws["A1"] = f"Global sensitivity analysis - mismatch index - case {case_name}"
    ws["A1"].font = title_font
    ws["A2"] = "sigma = abs( E_eff/(1-nu_eff) * (eps_T + eps_h) )"
    ws["A2"].font = Font(name=ARIAL, italic=True)

    ws["A4"] = "Variable inputs (varied simultaneously)"
    ws["A4"].font = bold
    write_header(ws, 5, ["Variable", "Units", "Lower bound", "Upper bound",
                         "Distribution"], [24, 12, 16, 16, 14])
    r = 6
    for n, u, b, d in zip(names, problem["units"], problem["bounds"],
                          problem["dists"]):
        for c, v in enumerate([n, u, float(b[0]), float(b[1]), d], start=1):
            ws.cell(row=r, column=c, value=v).font = body
        r += 1

    r += 1
    ws.cell(row=r, column=1, value="Fixed inputs (held constant)").font = bold
    r += 1
    write_header(ws, r, ["Parameter", "Value", "Units"])
    r += 1
    for key, val in fixed.items():
        ws.cell(row=r, column=1, value=key).font = body
        ws.cell(row=r, column=2, value=float(val)).font = body
        ws.cell(row=r, column=3, value=CONST_UNITS[key]).font = body
        r += 1

    r += 1
    write_header(ws, r, ["Sampling", "Value", "Note"])
    ws.column_dimensions["C"].width = 44
    r += 1
    for lab, val, note in [
        ("Base sample size N", N, "power of two, scrambled Sobol' sequence"),
        ("Number of variables k", k, ", ".join(names)),
        ("Model evaluations", N * (k + 2), "N*(k+2), Saltelli cross-sampling"),
        ("Bootstrap replicates", 500, "for the 95% confidence intervals"),
        ("Series terms", 60, "terms kept in the Fickian series solution"),
    ]:
        ws.cell(row=r, column=1, value=lab).font = body
        ws.cell(row=r, column=2, value=val).font = body
        ws.cell(row=r, column=3, value=note).font = body
        r += 1

    # ---------------- Samples --------------------------------------
    ws = wb.create_sheet("Samples")
    out_cols = [("Fo", "Fourier number D*t/h^2 (-)"),
                ("Mt_Minf", "Mt/Minf (-)"),
                ("C_moist", "Moisture conc. (kg/m^3)"),
                ("eps_h", "Hygroscopic strain (-)"),
                ("eps_T", "Thermal strain (-)"),
                ("sigma", "Mismatch index (MPa)")]
    headers = (["Run"] + [f"{n} ({u})" for n, u in zip(names, problem["units"])]
               + [lab for _, lab in out_cols] + ["Saltelli block"])
    write_header(ws, 1, headers, [8] + [16] * (len(headers) - 2) + [16])

    labels = block_labels(N, k, names)
    data = np.column_stack([X] + [out[key] for key, _ in out_cols])
    for i in range(data.shape[0]):
        row = i + 2
        ws.cell(row=row, column=1, value=i + 1)
        for c in range(data.shape[1]):
            ws.cell(row=row, column=c + 2, value=float(data[i, c]))
        ws.cell(row=row, column=data.shape[1] + 2, value=labels[i])

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{data.shape[0] + 1}"
    n_rows = data.shape[0]

    # ---------------- Output summary -------------------------------
    ws = wb.create_sheet("Output summary")
    ws["A1"] = "Statistics over every sampled combination (live formulas)"
    ws["A1"].font = title_font
    write_header(ws, 3, ["Quantity", "Min", "Mean", "Max", "Std dev"],
                 [32, 16, 16, 16, 16])
    quantities = [(f"{n} ({u})", get_column_letter(2 + i))
                  for i, (n, u) in enumerate(zip(names, problem["units"]))]
    quantities += [(lab, get_column_letter(k + 2 + j))
                   for j, (_, lab) in enumerate(out_cols)]
    r = 4
    for lab, col in quantities:
        rng_ = f"Samples!${col}$2:${col}${n_rows + 1}"
        ws.cell(row=r, column=1, value=lab).font = body
        ws.cell(row=r, column=2, value=f"=MIN({rng_})").font = body
        ws.cell(row=r, column=3, value=f"=AVERAGE({rng_})").font = body
        ws.cell(row=r, column=4, value=f"=MAX({rng_})").font = body
        ws.cell(row=r, column=5, value=f"=STDEV({rng_})").font = body
        r += 1

    # ---------------- Sobol S1 / ST --------------------------------
    lo1, hi1 = res["S1_ci"]
    loT, hiT = res["ST_ci"]
    ws = wb.create_sheet("Sobol S1 ST")
    ws["A1"] = f"First- and total-order Sobol indices for {Y_LABEL}"
    ws["A1"].font = title_font
    write_header(ws, 3, ["Variable", "S1", "S1 95% low", "S1 95% high",
                         "ST", "ST 95% low", "ST 95% high",
                         "ST - S1 (interaction share)"],
                 [16, 12, 14, 14, 12, 14, 14, 24])
    for i, n in enumerate(names):
        r = 4 + i
        vals = [n, res["S1"][i], lo1[i], hi1[i], res["ST"][i], loT[i], hiT[i]]
        for c, v in enumerate(vals, start=1):
            ws.cell(row=r, column=c,
                    value=v if c == 1 else float(v)).font = body
        ws.cell(row=r, column=8, value=f"=E{r}-B{r}").font = body

    tot = 4 + k
    ws.cell(row=tot, column=1, value="Sum").font = bold
    ws.cell(row=tot, column=2, value=f"=SUM(B4:B{tot - 1})").font = bold
    ws.cell(row=tot, column=5, value=f"=SUM(E4:E{tot - 1})").font = bold
    ws.cell(row=tot, column=8, value=f"=E{tot}-B{tot}").font = bold
    ws.cell(row=tot + 2, column=1,
            value="Sum ST - Sum S1 is the total interaction content; "
                  "Sum S1 well below 1 means a strongly non-additive model."
            ).font = Font(name=ARIAL, italic=True)

    # ---------------- Charts ---------------------------------------
    if png:
        try:
            from openpyxl.drawing.image import Image as XLImage
            wsc = wb.create_sheet("Charts")
            wsc["A1"] = "S1 / ST bar chart"
            wsc["A1"].font = title_font
            wsc.add_image(XLImage(png), "A3")
        except Exception as exc:
            print(f" (chart image not embedded: {exc})")

    # ---------------- How to read ----------------------------------
    ws = wb.create_sheet("How to read")
    ws.column_dimensions["A"].width = 12
    ws.column_dimensions["B"].width = 100
    ws["A1"] = "How to read this workbook"
    ws["A1"].font = title_font
    notes = [
        ("S1", "Fraction of output variance removed by fixing that variable "
               "alone. High S1 = strong direct driver."),
        ("ST", "S1 plus every interaction the variable takes part in. "
               "ST ~ 0 means the variable can be fixed anywhere in range."),
        ("ST - S1", "Large => the variable acts mainly through interactions."),
        ("Sum S1", "Well below 1 => the model is strongly non-additive."),
        ("Fixed", "Inputs not varied in this case are held at the values on "
                  "the Setup sheet; indices are conditional on those values."),
        ("Samples", "Every model evaluation: sampled inputs, intermediate "
                    "quantities and the mismatch index, plus the Saltelli "
                    "block (A, B, AB_i) each row came from."),
    ]
    r = 3
    for a, b in notes:
        ws.cell(row=r, column=1, value=a).font = bold
        cell = ws.cell(row=r, column=2, value=b)
        cell.font = body
        cell.alignment = Alignment(wrap_text=True, vertical="top")
        r += 1

    wb.save(fname)
    print(f" Excel workbook saved to {fname}  "
          f"({n_rows:,} sampled runs on the Samples sheet)")
    return fname


# ======================================================================
# 8. RUN ONE CASE
# ======================================================================
def run_case(case_name, vary, N):
    problem, fixed = make_problem(vary)
    names, k = problem["names"], len(problem["names"])

    print("\n" + "#" * 72)
    print(f" CASE {case_name}   (k = {k} variable inputs)")
    print("#" * 72)
    print("\n Variable inputs:")
    for n, u, b, d in zip(names, problem["units"], problem["bounds"],
                          problem["dists"]):
        print(f"   {n:<12} {b[0]:>11.4g}  to {b[1]:>11.4g}   [{u}]  ({d})")
    print("\n Constants (held fixed):")
    for key, val in fixed.items():
        print(f"   {key:<12} {val:>11.4g}   [{CONST_UNITS[key]}]")
    print(f"\n Model evaluations: N*(k+2) = {N}*({k + 2}) = {N * (k + 2):,}")

    X, _ = saltelli_sample(problem["bounds"], N, problem["dists"])
    out = model(X, names, fixed)
    summarise_outputs(out)

    res = sobol_indices(out[Y_KEY], N, k)
    report(res, names, f"case {case_name}: {Y_LABEL}")

    png = make_plots(res, names, f"Sensitivity - mismatch index ({case_name})",
                     f"sobol_indices_{case_name}.png")
    save_excel(X, out, res, problem, fixed, N, case_name, png,
               f"sobol_mismatch_{case_name}.xlsx")
    return res


# ======================================================================
# 9. MAIN
# ======================================================================
def main():
    print("=" * 72)
    print(" SOBOL SENSITIVITY - THERMAL/HYGROSCOPIC MISMATCH INDEX - CASE B")
    print("=" * 72)

    N = None
    if len(sys.argv) > 1:
        try:
            N = int(sys.argv[1])
        except ValueError:
            N = None
    if N is None:
        try:
            raw = input("\n Base sample size N (power of 2, blank = 4096): ").strip()
            N = int(raw) if raw else 4096
        except (ValueError, EOFError):
            N = 4096
    if N & (N - 1):
        N = 1 << (N - 1).bit_length()
        print(f" -> rounded up to the next power of two: N = {N}")

    for case_name, vary in CASES.items():
        run_case(case_name, vary, N)

    print("\n" + "=" * 72)
    print(" HOW TO READ THIS")
    print("=" * 72)
    print(" S1  fraction of output variance removed by fixing that variable")
    print("     alone. High S1 = strong direct driver.")
    print(" ST  S1 plus every interaction the variable takes part in.")
    print("     ST ~ 0 means the variable can be fixed at any value in range.")
    print(" Indices are conditional on the constant values set in CONSTANTS.")


if __name__ == "__main__":
    main()

 SOBOL SENSITIVITY - THERMAL/HYGROSCOPIC MISMATCH INDEX - CASE B



 Base sample size N (power of 2, blank = 4096):  



########################################################################
 CASE B_material   (k = 6 variable inputs)
########################################################################

 Variable inputs:
   D               2.15e-14  to       2e-12   [m^2/s]  (logunif)
   C_sat               92.5  to         366   [mol/m^3]  (unif)
   beta_h          0.000115  to     0.00038   [m^3/kg]  (unif)
   CTE_lam            8e-06  to     2.3e-05   [1/K]  (unif)
   E_eff            7.5e+09  to     2.8e+10   [Pa]  (unif)
   nu_eff             0.175  to        0.32   [-]  (unif)

 Constants (held fixed):
   Time           4.765e+05   [s]
   Thickness         0.0016   [m]
   dT                    50   [K]
   CTE_cu          1.65e-05   [1/K]
   M_water          0.01801   [kg/mol]

 Model evaluations: N*(k+2) = 4096*(8) = 32,768

 MONTE CARLO OUTPUT SUMMARY (all sampled combinations)
 Quantity                          min         mean          max          std
 -----------------------------------